# BrainModelKit test notebook

Use this notebook for quick, interactive checks while developing the package.

In [4]:
# %pip install git+https://github.com/Celso-RQ-Valle/BrainModelKit.git

In [5]:
import brainmodelkit

print(f"BrainModelKit version: {brainmodelkit.__version__}")

BrainModelKit version: 0.2.0


## Train Model

In [ ]:
from pyspark.sql import SparkSession

from brainmodelkit.pyspark import simulate_credit_data
from brainmodelkit.training.pyspark import train_model

spark = SparkSession.builder.appName("BrainModelTraining").getOrCreate()
data, features = simulate_credit_data(spark, row_count=1000, feature_count=6)
train_df, oot_df = data.randomSplit([0.75, 0.25], seed=42)

result = train_model(
    run_name="spark_random_forest",
    target_col="default_flag",
    feature_cols=features,
    train_df=train_df,
    oot_df=oot_df,
    model="random_forest",
    model_params={"numTrees": 20, "maxDepth": 5, "seed": 42},
    df_scoring=oot_df.drop("default_flag"),
    save_model_to="none",  # Avoid Spark model persistence on Windows.
    n_tiles=10,
)

print(result.metrics)  # oot_ks, oot_auc, oot_gini
print(result.output_dir)
result.scoring_predictions.select(*features, "score").show(5)

# The fitted pipeline includes feature assembly and accepts raw features.
result.model.transform(oot_df.drop("default_flag")).show(5)

RuntimeError: Spark model persistence on Windows requires Hadoop winutils. Configure HADOOP_HOME with compatible bin/winutils.exe and native Hadoop libraries before starting Spark, then restart the Python process/notebook kernel. MLflow also uses Spark's native writer. Use save_model_to='none' to train without saving, or run Spark in a configured Linux/WSL environment. See README: Spark on Windows.

In [ ]:
from pyspark.sql import SparkSession

from brainmodelkit.pyspark import simulate_credit_data
from brainmodelkit.training.pyspark import train_model

spark = SparkSession.builder.appName("BrainModelTraining").getOrCreate()
data, features = simulate_credit_data(spark, row_count=1000, feature_count=6)
train_df, oot_df = data.randomSplit([0.75, 0.25], seed=42)

result = train_model(
    run_name="spark_random_forest",
    target_col="default_flag",
    feature_cols=features,
    train_df=train_df,
    oot_df=oot_df,
    model="random_forest",
    model_params={"numTrees": 20, "maxDepth": 5, "seed": 42},
    df_scoring=oot_df.drop("default_flag"),  # Optional, no target required.
    save_model_to="none",  # First run: train/evaluate without model persistence.
    n_tiles=10,
)

print(result.metrics)  # oot_ks, oot_auc, oot_gini
print(result.output_dir)
result.scoring_predictions.select(*features, "score").show(5)

# The fitted pipeline includes feature assembly and accepts raw features.
result.model.transform(oot_df.drop("default_flag")).show(5)

{'oot_ks': 0.2989235046695658, 'oot_auc': 0.695052398944892, 'oot_gini': 0.390104797889784}
None
+----------+----------+----------+----------+----------+----------+-------------------+
|feature_01|feature_02|feature_03|feature_04|feature_05|feature_06|              score|
+----------+----------+----------+----------+----------+----------+-------------------+
| -1.007704|  0.172788|  1.216684|  0.819344| -0.627135|  1.190908|0.12477904230049726|
|  0.741722| -0.092543| -0.933112| -0.311408|  1.050733| -0.186687|0.29407065763323526|
| -1.432451|  0.586854|  1.944165|  1.858754| -0.573209| -0.092394| 0.3731952817954918|
| -1.017432|  0.558849| -0.866848|  0.384706|  0.861987| -1.073846| 0.6182265725759225|
| -0.127588| -1.497353|  0.332318| -0.267337| -0.216959| -0.259115|  0.347698961698494|
+----------+----------+----------+----------+----------+----------+-------------------+
only showing top 5 rows
+------------------+--------------+--------------------+----------------+------------+-